In [1]:
import json
import pandas as pd

In [2]:
# Read experiment file
file = json.load(open("../data/raw/experiment-processed-keerthi.json"))

In [3]:
file.keys()

dict_keys(['manifest', 'experiment', 'content', 'gazeSamples', 'materialSummaries'])

In [16]:
#235217fae96e47708c30c9e6b59cf7cb

file['experiment']['run']

{'sourceExperimentSetupId': '235217fae96e47708c30c9e6b59cf7cb',
 'sourceExperimentSetupName': 'Katarina and Keerthi Experiment',
 'isOneOff': False,
 'orderMode': 'fixed',
 'materials': [{'id': '1763d7577ad54ce2ab2fa0e7b77625cc',
   'order': 0,
   'title': 'easy text',
   'markdown': '# Lady and the Tramp\n\nLady and the Tramp is a 1955 American animated musical romantic comedy film produced by Walt Disney Productions and released by Buena Vista Film Distribution. Based on Ward Greene\'s 1945 Cosmopolitan magazine story Happy Dan, the Cynical Dog, it was directed by Hamilton Luske, Clyde Geronimi, and Wilfred Jackson. The film features the voices of Peggy Lee, Barbara Luddy, Larry Roberts, Bill Thompson, Bill Baucom, Stan Freberg, Verna Felton, Alan Reed, George Givot, Dallas McKennon, and Lee Millar. The film follows Lady, the pampered cocker spaniel, as she grows from puppy to adult, deals with changes in her family, and meets and falls in love with the homeless mutt Tramp.\n\nThe in

In [4]:
file['experiment']['lifecycleEvents'] # contains session start and session stop events with timestamps

file['experiment']['screen'] # contains screen information

file['content']['markdown'] # contains text as md

file['gazeSamples'] # actual data we are interested in
print('Elements in each sequence:\t', list(file['gazeSamples'][1000].keys()))
print('Elements in each gaze sample:\t', list(file['gazeSamples'][1000]['left'].keys()))
print('Elements in each focus area when user is not looking at the screen,\n \t\t\t\t i.e., "isInsideReadingArea=False":\t', list(file['gazeSamples'][0]['focus'].keys()))
print('Elements in each focus area when user is looking at the screen,\n \t\t\t\t  i.e., "isInsideReadingArea=True":\t', list(file['gazeSamples'][1000]['focus'].keys()))

Elements in each sequence:	 ['sequenceNumber', 'capturedAtUnixMs', 'deviceTimeStampUs', 'systemTimeStampUs', 'left', 'right', 'focus', 'materialRunId', 'materialIndex']
Elements in each gaze sample:	 ['gazePoint2D', 'gazePoint3D', 'pupil', 'gazeOrigin3D', 'gazeOriginTrackBox']
Elements in each focus area when user is not looking at the screen,
 				 i.e., "isInsideReadingArea=False":	 ['isInsideReadingArea', 'normalizedContentX', 'normalizedContentY', 'updatedAtUnixMs']
Elements in each focus area when user is looking at the screen,
 				  i.e., "isInsideReadingArea=True":	 ['isInsideReadingArea', 'normalizedContentX', 'normalizedContentY', 'activeTokenId', 'activeBlockId', 'activeSentenceId', 'updatedAtUnixMs', 'activeTokenText']


In [5]:
# Turn nested json into dataframe
df = pd.json_normalize(file, sep='_')
df;

In [6]:
# subset dataframe to not include the text in all rows after joining with gaze samples
df_subset = df[['experiment_sessionId', 'experiment_participant_name',
                'experiment_screen_screenWidthPx', 'experiment_screen_screenHeightPx',
                'experiment_screen_availableScreenWidthPx', 'experiment_screen_availableScreenHeightPx',
                'experiment_screen_physicalScreenWidthPx', 'experiment_screen_physicalScreenHeightPx',
                'experiment_screen_devicePixelRatio']]

In [7]:
# Flatten gaze samples
gaze_df = pd.json_normalize(file['gazeSamples'], sep='_')
gaze_df;

In [8]:
# cross join subset dataframe with gaze samples
df_cross_joined = df_subset.merge(gaze_df, how='cross')
df_cross_joined;

In [9]:
# Rename columns to fit et_file
df_cross_joined.rename(columns={
    'experiment_sessionId': 'Recording',
    'experiment_participant_name': 'Filename',
    'experiment_screen_screenWidthPx': 'screen_width_px',
    'experiment_screen_screenHeightPx': 'screen_height_px',
    'experiment_screen_availableScreenWidthPx': 'available_screen_width_px',
    'experiment_screen_availableScreenHeightPx': 'available_screen_height_px',
    'experiment_screen_physicalScreenWidthPx': 'physical_screen_width_px',
    'experiment_screen_physicalScreenHeightPx': 'physical_screen_height_px',
    'experiment_screen_devicePixelRatio': 'device_pixel_ratio',

    'sequenceNumber': 'row_number',
    'capturedAtUnixMs': 'time_stamps',
    'deviceTimeStampUs': 'device_time_stamp',

    'left_gazePoint2D_x':'left_gaze_point_on_display_area_0',
    'left_gazePoint2D_y': 'left_gaze_point_on_display_area_1',
    'right_gazePoint2D_x': 'right_gaze_point_on_display_area_0',
    'right_gazePoint2D_y': 'right_gaze_point_on_display_area_1',
    'left_gazePoint2D_validity': 'left_gaze_point_validity',
    'right_gazePoint2D_validity': 'right_gaze_point_validity',

    'left_gazePoint3D_x': 'left_gaze_point_in_user_coordinate_system_0',
    'left_gazePoint3D_y': 'left_gaze_point_in_user_coordinate_system_1',
    'left_gazePoint3D_z': 'left_gaze_point_in_user_coordinate_system_2',
    'right_gazePoint3D_x': 'right_gaze_point_in_user_coordinate_system_0',
    'right_gazePoint3D_y': 'right_gaze_point_in_user_coordinate_system_1',
    'right_gazePoint3D_z': 'right_gaze_point_in_user_coordinate_system_2',

    'left_gazeOrigin3D_x': 'left_gaze_origin_in_user_coordinate_system_0',
    'left_gazeOrigin3D_y': 'left_gaze_origin_in_user_coordinate_system_1',
    'left_gazeOrigin3D_z': 'left_gaze_origin_in_user_coordinate_system_2',
    'right_gazeOrigin3D_x': 'right_gaze_origin_in_user_coordinate_system_0',
    'right_gazeOrigin3D_y': 'right_gaze_origin_in_user_coordinate_system_1',
    'right_gazeOrigin3D_z': 'right_gaze_origin_in_user_coordinate_system_2',
    'left_gazeOrigin3D_validity': 'left_gaze_origin_validity',
    'right_gazeOrigin3D_validity': 'right_gaze_origin_validity',

    'left_gazeOriginTrackBox_x': 'left_gaze_origin_in_trackbox_coordinate_system_0',
    'left_gazeOriginTrackBox_y': 'left_gaze_origin_in_trackbox_coordinate_system_1',
    'left_gazeOriginTrackBox_z': 'left_gaze_origin_in_trackbox_coordinate_system_2',
    'right_gazeOriginTrackBox_x': 'right_gaze_origin_in_trackbox_coordinate_system_0',
    'right_gazeOriginTrackBox_y': 'right_gaze_origin_in_trackbox_coordinate_system_1',
    'right_gazeOriginTrackBox_z': 'right_gaze_origin_in_trackbox_coordinate_system_2',

    'left_pupil_diameterMm': 'left_pupil_diameter',
    'right_pupil_diameterMm': 'right_pupil_diameter',

    'focus_activeTokenId': 'token_id',
    'focus_activeTokenText': 'word'
}, inplace=True)

In [10]:
df_cross_joined.to_csv("../data/processed/experiment_gaze_samples.csv", index=False)

In [11]:
gaze_df

,sequenceNumber,capturedAtUnixMs,deviceTimeStampUs,systemTimeStampUs,materialRunId,materialIndex,left_gazePoint2D_x,left_gazePoint2D_y,left_gazePoint2D_validity,left_gazePoint3D_x,...,right_gazeOriginTrackBox_y,right_gazeOriginTrackBox_z,focus_isInsideReadingArea,focus_normalizedContentX,focus_normalizedContentY,focus_updatedAtUnixMs,focus_activeTokenId,focus_activeBlockId,focus_activeSentenceId,focus_activeTokenText
0,1,1778589790614,15461435137,1744916839276,1763d7577ad54ce2ab2fa0e7b77625cc,0,0.361087,0.241932,Valid,-82.931190,...,0.309021,0.566547,True,0.116124,0.208545,1778589790616,NaN,NaN,NaN,NaN
1,2,1778589790629,15461479550,1744916883689,1763d7577ad54ce2ab2fa0e7b77625cc,0,0.365505,0.243243,Valid,-80.293335,...,0.309960,0.567087,True,0.116124,0.208545,1778589790630,NaN,NaN,NaN,NaN
2,3,1778589790639,15461490653,1744916894792,1763d7577ad54ce2ab2fa0e7b77625cc,0,0.365330,0.242413,Valid,-80.398220,...,0.309940,0.567069,True,0.116124,0.208545,1778589790640,NaN,NaN,NaN,NaN
3,4,1778589790667,15461523962,1744916928101,1763d7577ad54ce2ab2fa0e7b77625cc,0,0.361264,0.247420,Valid,-82.825420,...,0.309424,0.567117,True,0.116124,0.208545,1778589790668,NaN,NaN,NaN,NaN
4,5,1778589790707,15461557272,1744916961411,1763d7577ad54ce2ab2fa0e7b77625cc,0,0.361329,0.250415,Valid,-82.786350,...,0.309136,0.568106,True,0.116463,0.209692,1778589790709,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87001,87002,1778591025889,16696724412,1746152128245,2b6070dd57954d56898a7864d397035c,1,0.456508,0.625594,Valid,-25.964684,...,0.384353,0.613818,True,0.391048,0.619901,1778591038645,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,Ways.
87002,87003,1778591025903,16696735515,1746152139348,2b6070dd57954d56898a7864d397035c,1,0.455511,0.624308,Valid,-26.560080,...,0.384361,0.613850,True,0.391048,0.619901,1778591038646,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,some
87003,87004,1778591025910,16696746618,1746152150451,2b6070dd57954d56898a7864d397035c,1,0.461409,0.621112,Valid,-23.039040,...,0.384409,0.613949,True,0.394559,0.619628,1778591038649,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,some
87004,87005,1778591025937,16696768825,1746152172658,2b6070dd57954d56898a7864d397035c,1,0.489530,0.616899,Valid,-6.250521,...,0.384428,0.614597,True,0.423827,0.617413,1778591038679,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,235217fae96e47708c30c9e6b59cf7cb:2b6070dd57954...,some
